In [ ]:
import os, subprocess, platform, sys

sections = []

sections.append('=' * 60 + '\nSYSTEM INFO\n' + '=' * 60)
sections.append(f'Platform: {platform.platform()}')
sections.append(f'Processor: {platform.processor()}')
sections.append(f'Architecture: {platform.machine()}')
sections.append(f'Python: {sys.version}')
sections.append(f'Byte order: {sys.byteorder}')

sections.append('=' * 60 + '\nOS / KERNEL\n' + '=' * 60)
sections.append(subprocess.getoutput('uname -a'))
sections.append(subprocess.getoutput('cat /etc/os-release'))
sections.append(subprocess.getoutput('hostnamectl 2>/dev/null || echo hostnamectl not available'))

sections.append('=' * 60 + '\nCPU INFO\n' + '=' * 60)
sections.append(subprocess.getoutput('cat /proc/cpuinfo'))
sections.append(subprocess.getoutput('lscpu'))

sections.append('=' * 60 + '\nMEMORY\n' + '=' * 60)
sections.append(subprocess.getoutput('free -h'))
sections.append(subprocess.getoutput('cat /proc/meminfo'))

sections.append('=' * 60 + '\nGPU INFO\n' + '=' * 60)
sections.append(subprocess.getoutput('nvidia-smi'))
sections.append(subprocess.getoutput('nvidia-smi -q'))

sections.append('=' * 60 + '\nDISK & MOUNTS\n' + '=' * 60)
sections.append(subprocess.getoutput('df -h'))
sections.append(subprocess.getoutput('mount | head -30'))
sections.append(subprocess.getoutput('lsblk 2>/dev/null || echo lsblk not available'))

sections.append('=' * 60 + '\nNETWORK\n' + '=' * 60)
sections.append(subprocess.getoutput('ip addr 2>/dev/null || ifconfig 2>/dev/null || echo no network tools'))
sections.append(subprocess.getoutput('cat /etc/resolv.conf 2>/dev/null || echo no resolv.conf'))

sections.append('=' * 60 + '\nCUDA INFO\n' + '=' * 60)
sections.append(subprocess.getoutput("nvcc --version 2>/dev/null || echo 'nvcc not found'"))
sections.append(subprocess.getoutput('ls /usr/local/cuda/lib64/ 2>/dev/null | head -30'))
sections.append(subprocess.getoutput('cat /usr/local/cuda/version.json 2>/dev/null || echo no version.json'))

sections.append('=' * 60 + '\nPYTORCH GPU\n' + '=' * 60)
try:
    import torch
    sections.append(f'PyTorch: {torch.__version__}')
    sections.append(f'CUDA available: {torch.cuda.is_available()}')
    sections.append(f'cuDNN version: {torch.backends.cudnn.version()}')
    sections.append(f'cuDNN enabled: {torch.backends.cudnn.enabled}')
    if torch.cuda.is_available():
        sections.append(f'Device count: {torch.cuda.device_count()}')
        for i in range(torch.cuda.device_count()):
            sections.append(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
            props = torch.cuda.get_device_properties(i)
            mem = getattr(props, 'total_global_mem', None) or getattr(props, 'total_mem', None)
            if mem:
                sections.append(f'  Memory: {mem / 1e9:.1f} GB')
            sections.append(f'  Compute capability: {props.major}.{props.minor}')
            sections.append(f'  Multi-processor count: {props.multi_processor_count}')
except ImportError:
    sections.append('PyTorch not installed')

sections.append('=' * 60 + '\nTENSORFLOW\n' + '=' * 60)
try:
    import tensorflow as tf
    sections.append(f'TensorFlow: {tf.__version__}')
    sections.append(f"GPUs: {tf.config.list_physical_devices('GPU')}")
    sections.append(f"CPUs: {tf.config.list_physical_devices('CPU')}")
except ImportError:
    sections.append('TensorFlow not installed')

sections.append('=' * 60 + '\nENVIRONMENT VARIABLES\n' + '=' * 60)
for k, v in sorted(os.environ.items()):
    sections.append(f'{k}={v}')

sections.append('=' * 60 + '\nTOOLS & COMPILERS\n' + '=' * 60)
for tool in ['gcc', 'g++', 'make', 'cmake', 'git', 'node', 'npm', 'java', 'R', 'julia', 'rustc', 'go']:
    ver = subprocess.getoutput(f'{tool} --version 2>/dev/null | head -1')
    sections.append(f'{tool}: {ver if ver else "not found"}')

sections.append('=' * 60 + '\nPYTHON PACKAGES\n' + '=' * 60)
sections.append(subprocess.getoutput('pip list 2>/dev/null'))

sections.append('=' * 60 + '\nKAGGLE INPUT DATA FILES\n' + '=' * 60)
sections.append(subprocess.getoutput('find /kaggle/input -type f 2>/dev/null'))
sections.append(subprocess.getoutput('ls -laR /kaggle/input/ 2>/dev/null || echo /kaggle/input is empty'))
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        path = os.path.join(dirname, filename)
        try:
            size = os.path.getsize(path)
            sections.append(f'{size:>12,} bytes  {path}')
        except:
            sections.append(f'  [error reading]  {path}')

sections.append('=' * 60 + '\nKAGGLE DIRECTORY STRUCTURE\n' + '=' * 60)
sections.append(subprocess.getoutput('ls -la /kaggle/'))
sections.append(subprocess.getoutput('ls -la /kaggle/working/'))
sections.append(subprocess.getoutput('ls -la /kaggle/temp/ 2>/dev/null || echo /kaggle/temp not found'))

sections.append('=' * 60 + '\nROOT FILESYSTEM TREE (depth 5)\n' + '=' * 60)
sections.append(subprocess.getoutput('find / -maxdepth 5 -type d 2>/dev/null | sort | head -500'))

sections.append('=' * 60 + '\nRUNTIME LIMITS\n' + '=' * 60)
sections.append(subprocess.getoutput('ulimit -a'))
sections.append(f'Recursion limit: {sys.getrecursionlimit()}')
sections.append(f'Max open files: {subprocess.getoutput("ulimit -n")}')
sections.append('CGROUP LIMITS:')
sections.append(subprocess.getoutput('cat /sys/fs/cgroup/memory/memory.limit_in_bytes 2>/dev/null || cat /sys/fs/cgroup/memory.max 2>/dev/null || echo no cgroup memory limit found'))
sections.append(subprocess.getoutput('cat /sys/fs/cgroup/cpu/cpu.cfs_quota_us 2>/dev/null || cat /sys/fs/cgroup/cpu.max 2>/dev/null || echo no cgroup cpu limit found'))

sections.append('=' * 60 + '\nINTERNET SPEED\n' + '=' * 60)
import time, urllib.request, socket

# Download test
url = 'http://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb'
start = time.perf_counter()
resp = urllib.request.urlopen(url)
data = resp.read()
dl_elapsed = time.perf_counter() - start
dl_mb = len(data) / 1e6
sections.append(f'Download: {dl_mb:.1f} MB in {dl_elapsed:.2f}s = {dl_mb/dl_elapsed:.1f} MB/s ({dl_mb*8/dl_elapsed:.1f} Mbps)')

# Upload test (POST to httpbin)
import http.client
upload_data = b'x' * (10 * 1024 * 1024)
start = time.perf_counter()
conn = http.client.HTTPSConnection('httpbin.org', timeout=30)
conn.request('POST', '/post', body=upload_data, headers={'Content-Type': 'application/octet-stream'})
resp = conn.getresponse()
resp.read()
conn.close()
ul_elapsed = time.perf_counter() - start
ul_mb = len(upload_data) / 1e6
sections.append(f'Upload: {ul_mb:.1f} MB in {ul_elapsed:.2f}s = {ul_mb/ul_elapsed:.1f} MB/s ({ul_mb*8/ul_elapsed:.1f} Mbps)')

# Latency test
sections.append(subprocess.getoutput('ping -c 5 8.8.8.8'))

# DNS resolution
start = time.perf_counter()
for host in ['google.com', 'kaggle.com', 'github.com', 'pypi.org', 'huggingface.co']:
    ip = socket.gethostbyname(host)
    sections.append(f'{host} -> {ip}')
dns_elapsed = time.perf_counter() - start
sections.append(f'DNS resolution (5 hosts): {dns_elapsed*1000:.0f} ms')

output = '\n'.join(sections)
print(output)
with open('/kaggle/working/output.txt', 'w') as f:
    f.write(output)

In [ ]:
# ============================================================
# C++ PORT TOOLCHAIN PROBE  (added for ARC C++/CUDA port)
#   1. C++20 + std::format   (libstdc++ landmine)
#   2. nvcc CUDA sm_75 compile+run (Tesla T4)
#   3. nvcc + C++20 std::format together (the real combo we need)
# Appends results to /kaggle/working/output.txt
# ============================================================
import subprocess, tempfile, os, textwrap

probe = []
def log(s):
    probe.append(str(s)); print(s)

log("=" * 60 + "\nC++ PORT TOOLCHAIN PROBE\n" + "=" * 60)

for tool in ["g++", "gcc", "nvcc", "cmake"]:
    log(tool + ": " + subprocess.getoutput(tool + " --version 2>&1 | head -1"))

work = tempfile.mkdtemp()

# TEST 1: C++20 std::format
log("\n" + "-" * 60 + "\nTEST 1: C++20 std::format\n" + "-" * 60)
p = os.path.join(work, "t1.cpp")
open(p, "w").write(textwrap.dedent("""
    #include <format>
    #include <string>
    #include <iostream>
    int main(){
        std::string s = std::format("{}-{:04d}-{:.2f}", "ok", 7, 3.14159);
        std::cout << s << std::endl;
        return s == "ok-0007-3.14" ? 0 : 2;
    }
"""))
r = subprocess.run(["g++", "-std=c++20", p, "-o", os.path.join(work, "t1")],
                    capture_output=True, text=True)
if r.returncode != 0:
    log("COMPILE: FAIL")
    fp = os.path.join(work, "fp.cpp")
    open(fp, "w").write("#include <version>\n#include <cstdio>\nint main(){"
                        "#ifdef __cpp_lib_format\nprintf(\"%ld\",(long)__cpp_lib_format);"
                        "#else\nprintf(\"UNDEFINED\");\n#endif\n}")
    subprocess.run(["g++", "-std=c++20", fp, "-o", os.path.join(work, "fp")],
                    capture_output=True, text=True)
    log("__cpp_lib_format = " + subprocess.getoutput(os.path.join(work, "fp")))
    log("stderr[:1200]:\n" + r.stderr[:1200])
else:
    rr = subprocess.run([os.path.join(work, "t1")], capture_output=True, text=True)
    log("COMPILE OK | run rc=%d out=%r" % (rr.returncode, rr.stdout.strip()))
    log("RESULT: std::format " + ("WORKS" if rr.returncode == 0 else "RUNTIME MISMATCH"))

# TEST 2: nvcc CUDA sm_75
log("\n" + "-" * 60 + "\nTEST 2: nvcc CUDA sm_75 compile+run\n" + "-" * 60)
p = os.path.join(work, "t2.cu")
open(p, "w").write(textwrap.dedent("""
    #include <cstdio>
    __global__ void k(int* d){ d[threadIdx.x] = threadIdx.x * threadIdx.x; }
    int main(){
        int *d, h[8];
        cudaMalloc(&d, 8*sizeof(int));
        k<<<1,8>>>(d);
        cudaError_t e = cudaDeviceSynchronize();
        if(e != cudaSuccess){ printf("CUDA_ERR:%s\\n", cudaGetErrorString(e)); return 3; }
        cudaMemcpy(h, d, 8*sizeof(int), cudaMemcpyDeviceToHost);
        printf("CUDA_OK:");
        for(int i=0;i<8;i++) printf("%d ", h[i]);
        printf("\\n");
        return (h[7]==49) ? 0 : 4;
    }
"""))
r = subprocess.run(["nvcc", "-arch=sm_75", "-std=c++17", p, "-o", os.path.join(work, "t2")],
                    capture_output=True, text=True)
if r.returncode != 0:
    log("NVCC COMPILE: FAIL\n" + r.stderr[:1500])
else:
    rr = subprocess.run([os.path.join(work, "t2")], capture_output=True, text=True)
    log("NVCC COMPILE OK | run rc=%d out=%r" % (rr.returncode, rr.stdout.strip()))
    log("RESULT: CUDA sm_75 " + ("WORKS" if rr.returncode == 0 else "FAILED"))

# TEST 3: nvcc + C++20 std::format together
log("\n" + "-" * 60 + "\nTEST 3: nvcc host-side C++20 std::format\n" + "-" * 60)
p = os.path.join(work, "t3.cu")
open(p, "w").write(textwrap.dedent("""
    #include <format>
    #include <string>
    #include <cstdio>
    __global__ void noop(){}
    int main(){
        std::string s = std::format("nvcc+fmt {}", 42);
        printf("%s\\n", s.c_str());
        noop<<<1,1>>>();
        return cudaDeviceSynchronize()==cudaSuccess ? 0 : 5;
    }
"""))
r = subprocess.run(["nvcc", "-arch=sm_75", "-std=c++20", p, "-o", os.path.join(work, "t3")],
                    capture_output=True, text=True)
if r.returncode != 0:
    log("NVCC+C++20 COMPILE: FAIL  <-- port must drop std::format or pass -ccbin")
    log(r.stderr[:1500])
else:
    rr = subprocess.run([os.path.join(work, "t3")], capture_output=True, text=True)
    log("NVCC+C++20 COMPILE OK | run rc=%d out=%r" % (rr.returncode, rr.stdout.strip()))
    log("RESULT: nvcc + std::format " + ("WORKS (port unblocked)" if rr.returncode == 0 else "RUNTIME FAIL"))

try:
    with open("/kaggle/working/output.txt", "a") as f:
        f.write("\n\n" + "\n".join(probe))
    log("\n[appended to /kaggle/working/output.txt]")
except Exception as e:
    log("could not append to output.txt: %r" % e)


In [ ]:
# ============================================================
# C++ STANDARD / FEATURE CAPABILITY PROBE (probe #2)
#   - max usable -std on this g++/nvcc
#   - which C++20 features compile (concepts/ranges/bit/jthread/etc.)
#   - is <format> available via ANY installed compiler/stdlib?
# Appends to /kaggle/working/output.txt
# ============================================================
import subprocess, tempfile, os, textwrap, shutil

probe = []
def log(s):
    probe.append(str(s)); print(s)

log("=" * 60 + "\nC++ STD / FEATURE CAPABILITY PROBE\n" + "=" * 60)
work = tempfile.mkdtemp()

def try_compile(name, code, compiler="g++", std="c++20", extra=None):
    src = os.path.join(work, name)
    open(src, "w").write(code)
    out = os.path.join(work, name + ".bin")
    cmd = [compiler, "-std=" + std, src, "-o", out] + (extra or [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        return False, r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "compile fail"
    rr = subprocess.run([out], capture_output=True, text=True)
    return rr.returncode == 0, rr.stdout.strip()

# 1. Max -std accepted by g++
log("\n-- max -std accepted (g++ 11.4) --")
for std in ["c++17", "c++20", "c++23", "c++2b", "c++26"]:
    ok, msg = try_compile("s_%s.cpp" % std.replace("+", "p"),
                           "int main(){return 0;}", std=std)
    log("  -std=%-7s : %s" % (std, "OK" if ok else "REJECTED (%s)" % msg))

# 2. List all compilers present
log("\n-- compilers present on image --")
for c in ["g++", "g++-11", "g++-12", "g++-13", "g++-14", "clang++",
          "clang++-14", "clang++-15", "clang++-16", "clang++-17"]:
    pth = shutil.which(c)
    if pth:
        log("  %-12s -> %s | %s" % (c, pth,
            subprocess.getoutput("%s --version 2>&1 | head -1" % c)))

# 3. <format> via any compiler / libc++
log("\n-- <format> availability sweep --")
fmt_code = ('#include <format>\n#include <cstdio>\n'
            'int main(){auto s=std::format("{}",42);printf("%s",s.c_str());return s=="42"?0:1;}')
for c in ["g++", "g++-13", "g++-14", "clang++", "clang++-16", "clang++-17"]:
    if not shutil.which(c): continue
    ok, msg = try_compile("fmt_%s.cpp" % c.replace("+", "p"), fmt_code, compiler=c, std="c++20")
    log("  %-10s -std=c++20            : %s" % (c, "WORKS" if ok else "no (%s)" % msg))
    if c.startswith("clang"):
        ok2, m2 = try_compile("fmtl_%s.cpp" % c.replace("+", "p"), fmt_code,
                               compiler=c, std="c++20", extra=["-stdlib=libc++"])
        log("  %-10s -std=c++20 -stdlib=libc++ : %s" % (c, "WORKS" if ok2 else "no (%s)" % m2))

# 4. Which C++20 features the core likely needs (besides std::format)
log("\n-- C++20 feature checks (g++ -std=c++20) --")
feats = {
 "concepts":        '#include <concepts>\ntemplate<std::integral T> T f(T x){return x;}\nint main(){return f(0);}',
 "ranges":          '#include <ranges>\n#include <vector>\nint main(){std::vector<int> v{1,2};auto r=v|std::views::reverse;return *r.begin()==2?0:1;}',
 "bit":             '#include <bit>\n#include <cstdint>\nint main(){return std::popcount((unsigned)7)==3?0:1;}',
 "span":            '#include <span>\nint main(){int a[3]{};std::span<int> s(a);return s.size()==3?0:1;}',
 "jthread":         '#include <thread>\nint main(){std::jthread t([]{});return 0;}',
 "designated_init": 'struct P{int a;int b;};int main(){P p{.a=1,.b=2};return p.b-2;}',
 "consteval":       'consteval int f(){return 7;}int main(){return f()-7;}',
 "starts_with":     '#include <string>\nint main(){std::string s="hello";return s.starts_with("he")?0:1;}',
 "to_address":      '#include <memory>\nint main(){int x=0;return std::to_address(&x)==&x?0:1;}',
 "source_location": '#include <source_location>\nint main(){auto l=std::source_location::current();return l.line()>0?0:1;}',
}
for name, code in feats.items():
    ok, msg = try_compile("f_%s.cpp" % name, code, std="c++20")
    log("  %-16s : %s" % (name, "OK" if ok else "MISSING (%s)" % msg))

try:
    with open("/kaggle/working/output.txt", "a") as f:
        f.write("\n\n" + "\n".join(probe))
    log("\n[appended to /kaggle/working/output.txt]")
except Exception as e:
    log("append failed: %r" % e)


In [ ]:
# ============================================================
# COMPILE CORE ON KAGGLE gcc 11 (first real Linux compile attempt)
#   - dataset attached at /kaggle/input/solver-source-code/
#   - try g++ -std=c++20 on each core header via a tiny .cpp
#   - try nvcc on cuda_solver.cu
#   - dump ALL errors (no -Werror, fcoutput tweaks for readable diagnostics)
# Appends to /kaggle/working/output.txt
# ============================================================
import subprocess, os, tempfile, zipfile, glob

log = []
def L(s):
    log.append(str(s)); print(s)

L("=" * 60 + "\nCOMPILE CORE ON KAGGLE gcc 11\n" + "=" * 60)

ds = "/kaggle/input/solver-source-code"
L("dataset contents:")
for p in sorted(glob.glob(ds + "/**", recursive=True))[:50]:
    L("  " + p)

# Some datasets land zipped (we used --dir-mode zip).
src_root = ds
zips = glob.glob(ds + "/*.zip")
if zips:
    work = "/kaggle/working/src"
    os.makedirs(work, exist_ok=True)
    for z in zips:
        L(f"unzipping {z} -> {work}")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(work)
    src_root = work
    L("after unzip:")
    for p in sorted(glob.glob(src_root + "/**", recursive=True))[:60]:
        L("  " + p)

inc = src_root + "/include"
tp  = inc + "/third_party"
cu  = src_root + "/src/cuda_solver.cu"

# Compile each core header by way of a tiny .cpp that includes it.
core_headers = [
    "environment.hpp", "global_app_state.hpp", "logger.hpp",
    "fixed_dsl.hpp", "dsl_executor.hpp", "puzzle_loader.hpp",
    "solver_beam.hpp", "solver_executor.hpp",
]
work = tempfile.mkdtemp()
for h in core_headers:
    L("\n" + "-" * 60 + f"\nTRY: g++ -c on {h}\n" + "-" * 60)
    src = os.path.join(work, h.replace(".", "_") + ".cpp")
    open(src, "w").write(f'#include "{h}"\nint main(){{}}\n')
    r = subprocess.run(
        ["g++", "-std=c++20", "-c", src, "-I", inc, "-I", tp,
         "-o", os.path.join(work, h + ".o")],
        capture_output=True, text=True,
    )
    L(f"  exit={r.returncode}")
    if r.stderr.strip():
        L("  stderr (first 1500 chars):")
        L("  " + r.stderr[:1500].replace("\n", "\n  "))

# nvcc on the CUDA TU
L("\n" + "-" * 60 + f"\nTRY: nvcc -c on cuda_solver.cu\n" + "-" * 60)
if os.path.exists(cu):
    r = subprocess.run(
        ["nvcc", "-arch=sm_75", "-std=c++17", "-c", cu, "-I", inc, "-I", tp,
         "-o", os.path.join(work, "cuda_solver.o")],
        capture_output=True, text=True,
    )
    L(f"  exit={r.returncode}")
    if r.stderr.strip():
        L("  stderr (first 1500 chars):")
        L("  " + r.stderr[:1500].replace("\n", "\n  "))
else:
    L("  cuda_solver.cu not found at " + cu)

try:
    with open("/kaggle/working/output.txt", "a") as f:
        f.write("\n\n" + "\n".join(log))
    L("\n[appended to /kaggle/working/output.txt]")
except Exception as e:
    L("append failed: %r" % e)
